# Day 41 — Transformer Encoder for API Sequence Classification

Small Transformer encoder on the same API call sequences as the Day 40 LSTM,
to compare stability and accuracy.

**Architecture:** Embedding + Positional Encoding → TransformerEncoder (2 layers, 4 heads) → Mean Pool → Linear(2)

**Key differences from LSTM:**
- Self-attention sees the whole sequence at once (no recurrence)
- Positional encoding adds order information explicitly
- Mean pooling over all positions (vs. last hidden state)
- Potentially better at long-range dependencies

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import json
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

from dynamic_sandbox.api_sequence import extract_flat_sequence
from models.tokenizer import APITokenizer

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Data (same as Day 40)

In [ ]:
fixtures_dir = os.path.join('..', 'tests', 'fixtures')

with open(os.path.join(fixtures_dir, 'cape_report_benign.json')) as f:
    benign_report = json.load(f)
with open(os.path.join(fixtures_dir, 'cape_report_suspicious.json')) as f:
    susp_report = json.load(f)

benign_apis = extract_flat_sequence(benign_report)
susp_apis = extract_flat_sequence(susp_report)

def augment_sequence(base_seq, n_samples=100, min_frac=0.5, max_frac=1.0, noise_rate=0.1):
    samples = []
    api_pool = list(set(base_seq))
    for _ in range(n_samples):
        frac = random.uniform(min_frac, max_frac)
        length = max(3, int(len(base_seq) * frac))
        seq = []
        for i in range(length):
            if i < len(base_seq) and random.random() > noise_rate:
                seq.append(base_seq[i % len(base_seq)])
            else:
                seq.append(random.choice(api_pool))
        samples.append(seq)
    return samples

N_PER_CLASS = 150
benign_samples = augment_sequence(benign_apis, N_PER_CLASS)
susp_samples = augment_sequence(susp_apis, N_PER_CLASS)

all_sequences = benign_samples + susp_samples
all_labels = [0] * N_PER_CLASS + [1] * N_PER_CLASS

MAX_LEN = 128
tokenizer = APITokenizer(max_len=MAX_LEN, min_freq=1)
tokenizer.build_vocab(all_sequences)
print(f'Vocab size: {tokenizer.vocab_size}, Max len: {MAX_LEN}')

indices = list(range(len(all_sequences)))
random.shuffle(indices)
split = int(0.8 * len(indices))
train_idx, test_idx = indices[:split], indices[split:]

In [ ]:
class APISeqDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer):
        self.encoded = [tokenizer.encode_padded(seq) for seq in sequences]
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (torch.tensor(self.encoded[idx], dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))

train_seqs = [all_sequences[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]
test_seqs = [all_sequences[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

train_loader = DataLoader(APISeqDataset(train_seqs, train_labels, tokenizer),
                          batch_size=32, shuffle=True)
test_loader = DataLoader(APISeqDataset(test_seqs, test_labels, tokenizer),
                         batch_size=32)
print(f'Train: {len(train_seqs)}, Test: {len(test_seqs)}')

## 2. Transformer Model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, num_classes=2, dropout=0.1, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        pad_mask = (x == self.pad_idx)  # (batch, seq_len)
        emb = self.embedding(x) * math.sqrt(self.d_model)
        emb = self.pos_encoder(emb)
        encoded = self.transformer_encoder(emb, src_key_padding_mask=pad_mask)

        # Mean pooling over non-pad positions
        mask_expanded = (~pad_mask).unsqueeze(-1).float()  # (batch, seq_len, 1)
        summed = (encoded * mask_expanded).sum(dim=1)
        counts = mask_expanded.sum(dim=1).clamp(min=1)
        pooled = summed / counts  # (batch, d_model)

        return self.fc(pooled)


model = TransformerClassifier(
    vocab_size=tokenizer.vocab_size,
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    dropout=0.1,
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

## 3. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

EPOCHS = 30
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for seqs, labels in train_loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(seqs)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(labels)

    scheduler.step()
    train_loss = total_loss / total
    train_acc = correct / total
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for seqs, labels in test_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            logits = model(seqs)
            loss = criterion(logits, labels)
            val_loss += loss.item() * len(labels)
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_total += len(labels)

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:>2}/{EPOCHS}  '
              f'train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  '
              f'val_loss={val_loss:.4f}  val_acc={val_acc:.3f}')

## 4. Evaluation

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

model.eval()
all_preds, all_probs, all_true = [], [], []

with torch.no_grad():
    for seqs, labels in test_loader:
        seqs = seqs.to(device)
        logits = model(seqs)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_probs.extend(probs)
        all_true.extend(labels.numpy())

print('=== Transformer Classification Report ===')
print(classification_report(all_true, all_preds,
                            target_names=['benign', 'malicious']))

try:
    auc = roc_auc_score(all_true, all_probs)
    print(f'AUC-ROC: {auc:.4f}')
except ValueError:
    print('AUC-ROC: could not compute')

cm = confusion_matrix(all_true, all_preds)
print(f'\nConfusion Matrix:\n{cm}')

## 5. LSTM vs Transformer Comparison

| Metric | LSTM (Day 40) | Transformer (Day 41) | Notes |
|--------|:---:|:---:|-------|
| Parameters | ~175K | ~115K | Transformer is smaller with these settings |
| Training stability | Stable | May oscillate early | Self-attention warm-up can help |
| Convergence speed | Fast | Comparable | Both reach high accuracy on synthetic data |
| Long-range deps | Struggles >200 tokens | Handles well | Attention sees all positions equally |
| Inference speed | Sequential | Parallelizable | Transformer benefits from GPU batching |

### Expected behavior on real data:
- **Short sequences (<100 calls):** LSTM and Transformer perform similarly
- **Medium sequences (100-500 calls):** Transformer may edge ahead
- **Long sequences (500-2048 calls):** Transformer significantly better at
  capturing long-range patterns (e.g., persistence setup → C2 callback 400 calls later)

### Recommendation for fusion (Track 4b):
Use the Transformer as the sequence encoder in learned fusion — it gives a
richer sequence embedding that complements the static feature vector.